# Problema do Caixeiro Viajante (TSP) com Têmpera Simulada e Algoritmo Genético

**Tarefa 1 — ICSI30 (Sistemas Inteligentes)**

Implementação de dois algoritmos de busca local/evolutiva aplicados ao TSP:
- **Têmpera Simulada (Simulated Annealing)**: busca mono-solução com aceitação probabilística
- **Algoritmo Genético (AG)**: busca por população com recombinação e mutação

Baseado em: Russell & Norvig (2020), Cap. 4 § 4.1

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import random
from typing import List, Tuple
import math

# Configuração de aleatoreidade para reprodutibilidade
random.seed(42)
np.random.seed(42)

print("Bibliotecas carregadas com sucesso.")

## 1. Definição do Problema TSP

### Formulação
- **Estado**: permutação de cidades [c₁, c₂, ..., cₙ]
- **Função objetivo**: minimizar distância total de viagem
- **Vizinhança**: 2-opt (trocar dois pares de cidades)
- **Restrição**: cada cidade visitada exatamente uma vez

In [ ]:
class TSPInstance:
    """Instância do Problema do Caixeiro Viajante."""
    
    def __init__(self, cities: np.ndarray, name: str = "TSP"):
        """
        Args:
            cities: array (n, 2) com coordenadas (x, y) das cidades
            name: nome da instância
        """
        self.cities = cities
        self.n_cities = len(cities)
        self.name = name
        self._compute_distances()
    
    def _compute_distances(self):
        """Pré-computa matriz de distâncias euclidianas."""
        self.distances = np.zeros((self.n_cities, self.n_cities))
        for i in range(self.n_cities):
            for j in range(self.n_cities):
                if i != j:
                    dx = self.cities[i][0] - self.cities[j][0]
                    dy = self.cities[i][1] - self.cities[j][1]
                    self.distances[i][j] = np.sqrt(dx**2 + dy**2)
    
    def evaluate(self, tour: List[int]) -> float:
        """Calcula o custo total de um tour (menor é melhor)."""
        cost = 0.0
        for i in range(len(tour)):
            from_city = tour[i]
            to_city = tour[(i + 1) % len(tour)]
            cost += self.distances[from_city][to_city]
        return cost
    
    def random_tour(self) -> List[int]:
        """Gera um tour aleatório."""
        tour = list(range(self.n_cities))
        random.shuffle(tour)
        return tour


# Criar instância de exemplo: 10 cidades aleatórias
np.random.seed(42)
cities = np.random.rand(10, 2) * 100
tsp = TSPInstance(cities, name="TSP-10-aleatorio")

print(f"Instância criada: {tsp.name}")
print(f"Número de cidades: {tsp.n_cities}")
print(f"Espaço de buscas: {math.factorial(tsp.n_cities):,} permutações")

## 2. Têmpera Simulada (Simulated Annealing)

### Algoritmo
1. Começar em um tour aleatório
2. A cada iteração:
   - Gerar vizinho (2-opt: trocar dois pares)
   - Se melhora: aceitar
   - Senão: aceitar com probabilidade e^(ΔE/T)
   - Reduzir temperatura T segundo agenda
3. Parar quando T → 0 ou critério de convergência

In [ ]:
class SimulatedAnnealing:
    """Têmpera Simulada para TSP."""
    
    def __init__(self, tsp: TSPInstance, initial_temp: float = 1000.0, 
                 cooling_rate: float = 0.95, max_iterations: int = 1000):
        self.tsp = tsp
        self.initial_temp = initial_temp
        self.cooling_rate = cooling_rate
        self.max_iterations = max_iterations
        self.best_tour = None
        self.best_cost = float('inf')
        self.history = {"cost": [], "temp": [], "iteration": []}
    
    def _2opt_neighbor(self, tour: List[int]) -> List[int]:
        """Gera vizinho via 2-opt: trocar dois arcos."""
        new_tour = tour.copy()
        i, j = random.sample(range(len(tour)), 2)
        if i > j:
            i, j = j, i
        # Reverter segmento entre i e j (2-opt clássico)
        new_tour[i:j+1] = reversed(new_tour[i:j+1])
        return new_tour
    
    def run(self, verbose: bool = True):
        """Executa Têmpera Simulada."""
        current_tour = self.tsp.random_tour()
        current_cost = self.tsp.evaluate(current_tour)
        
        self.best_tour = current_tour.copy()
        self.best_cost = current_cost
        
        temperature = self.initial_temp
        
        for iteration in range(self.max_iterations):
            if temperature < 1e-3:
                if verbose:
                    print(f"Converged at iteration {iteration}")
                break
            
            # Gerar vizinho
            neighbor_tour = self._2opt_neighbor(current_tour)
            neighbor_cost = self.tsp.evaluate(neighbor_tour)
            
            # Delta de custo (negativo = melhoria)
            delta = neighbor_cost - current_cost
            
            # Critério de aceitação
            if delta < 0 or random.random() < np.exp(-delta / temperature):
                current_tour = neighbor_tour
                current_cost = neighbor_cost
            
            # Atualizar melhor encontrado
            if current_cost < self.best_cost:
                self.best_tour = current_tour.copy()
                self.best_cost = current_cost
            
            # Log
            self.history["cost"].append(self.best_cost)
            self.history["temp"].append(temperature)
            self.history["iteration"].append(iteration)
            
            # Agenda de resfriamento
            temperature *= self.cooling_rate
        
        if verbose:
            print(f"\n✓ Têmpera Simulada concluída")
            print(f"  Melhor custo encontrado: {self.best_cost:.2f}")
            print(f"  Tour: {self.best_tour}")
        
        return self.best_tour, self.best_cost


# Executar Têmpera Simulada
sa = SimulatedAnnealing(tsp, initial_temp=1000.0, cooling_rate=0.95, max_iterations=500)
best_tour_sa, best_cost_sa = sa.run(verbose=True)

## 3. Algoritmo Genético

### Componentes
- **Cromossomo**: permutação de cidades [c₁, c₂, ..., cₙ]
- **Aptidão**: inverso do custo (maior é melhor)
- **Cruzamento**: Partially Mapped Crossover (PMX)
- **Mutação**: inversão de segmento ou troca de posição

In [ ]:
class GeneticAlgorithm:
    """Algoritmo Genético para TSP."""
    
    def __init__(self, tsp: TSPInstance, population_size: int = 50, 
                 mutation_rate: float = 0.1, max_generations: int = 100):
        self.tsp = tsp
        self.population_size = population_size
        self.mutation_rate = mutation_rate
        self.max_generations = max_generations
        self.best_tour = None
        self.best_cost = float('inf')
        self.history = {"best_cost": [], "avg_cost": [], "generation": []}
    
    def _fitness(self, tour: List[int]) -> float:
        """Calcula aptidão: 1 / (1 + custo)."""
        cost = self.tsp.evaluate(tour)
        return 1.0 / (1.0 + cost)
    
    def _pmx_crossover(self, parent1: List[int], parent2: List[int]) -> List[int]:
        """Cruzamento PMX (Partially Mapped Crossover)."""
        n = len(parent1)
        p1, p2 = random.sample(range(n), 2)
        if p1 > p2:
            p1, p2 = p2, p1
        
        child = [-1] * n
        
        # Copiar segmento de parent1
        child[p1:p2+1] = parent1[p1:p2+1]
        
        # Preencher resto com parent2
        for i in range(n):
            if i < p1 or i > p2:
                city = parent2[i]
                if city not in child:
                    child[i] = city
        
        # Resolver conflitos
        for i in range(n):
            if child[i] == -1:
                for city in range(n):
                    if city not in child:
                        child[i] = city
                        break
        
        return child
    
    def _mutate(self, tour: List[int]) -> List[int]:
        """Mutação: inversão de segmento."""
        if random.random() < self.mutation_rate:
            new_tour = tour.copy()
            i, j = random.sample(range(len(tour)), 2)
            if i > j:
                i, j = j, i
            new_tour[i:j+1] = reversed(new_tour[i:j+1])
            return new_tour
        return tour
    
    def run(self, verbose: bool = True):
        """Executa Algoritmo Genético."""
        # Inicializar população
        population = [self.tsp.random_tour() for _ in range(self.population_size)]
        
        for generation in range(self.max_generations):
            # Avaliar aptidão
            fitnesses = [self._fitness(tour) for tour in population]
            costs = [self.tsp.evaluate(tour) for tour in population]
            
            # Atualizar melhor
            best_idx = np.argmin(costs)
            if costs[best_idx] < self.best_cost:
                self.best_tour = population[best_idx].copy()
                self.best_cost = costs[best_idx]
            
            # Log
            self.history["best_cost"].append(self.best_cost)
            self.history["avg_cost"].append(np.mean(costs))
            self.history["generation"].append(generation)
            
            # Seleção e reprodução
            fitness_sum = sum(fitnesses)
            if fitness_sum > 0:
                probs = [f / fitness_sum for f in fitnesses]
            else:
                probs = [1.0 / len(population)] * len(population)
            
            new_population = []
            while len(new_population) < self.population_size:
                # Seleção por roleta
                parent1 = population[np.random.choice(len(population), p=probs)]
                parent2 = population[np.random.choice(len(population), p=probs)]
                
                # Cruzamento
                child = self._pmx_crossover(parent1, parent2)
                
                # Mutação
                child = self._mutate(child)
                
                new_population.append(child)
            
            population = new_population
        
        if verbose:
            print(f"\n✓ Algoritmo Genético concluído")
            print(f"  Melhor custo encontrado: {self.best_cost:.2f}")
            print(f"  Tour: {self.best_tour}")
        
        return self.best_tour, self.best_cost


# Executar Algoritmo Genético
ga = GeneticAlgorithm(tsp, population_size=30, mutation_rate=0.1, max_generations=100)
best_tour_ga, best_cost_ga = ga.run(verbose=True)

## 4. Visualização dos Resultados

In [ ]:
def plot_tsp_tour(tsp: TSPInstance, tour: List[int], title: str = "TSP Tour", cost: float = None):
    """Plota um tour do TSP."""
    fig, ax = plt.subplots(figsize=(8, 8))
    
    # Plotar cidades
    ax.scatter(tsp.cities[:, 0], tsp.cities[:, 1], c='red', s=100, zorder=3)
    
    # Plotar tour
    tour_points = tsp.cities[tour + [tour[0]]]  # Fechar o ciclo
    ax.plot(tour_points[:, 0], tour_points[:, 1], 'b-', linewidth=2, zorder=1)
    
    # Anotações
    for i, (x, y) in enumerate(tsp.cities):
        ax.annotate(str(i), (x, y), xytext=(5, 5), textcoords='offset points', fontsize=9)
    
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    if cost is not None:
        ax.set_title(f"{title} (Custo: {cost:.2f})")
    else:
        ax.set_title(title)
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal')
    plt.tight_layout()
    return fig


# Plotar tours
fig1 = plot_tsp_tour(tsp, best_tour_sa, title="Têmpera Simulada", cost=best_cost_sa)
plt.show()

fig2 = plot_tsp_tour(tsp, best_tour_ga, title="Algoritmo Genético", cost=best_cost_ga)
plt.show()

## 5. Convergência Comparativa

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Têmpera Simulada
ax1 = axes[0]
ax1.plot(sa.history["iteration"], sa.history["cost"], 'b-', linewidth=2, label='Melhor custo')
ax1.set_xlabel('Iteração')
ax1.set_ylabel('Custo')
ax1.set_title(f'Têmpera Simulada (Melhor: {sa.best_cost:.2f})')
ax1.grid(True, alpha=0.3)
ax1.legend()

# Algoritmo Genético
ax2 = axes[1]
ax2.plot(ga.history["generation"], ga.history["best_cost"], 'r-', linewidth=2, label='Melhor custo')
ax2.plot(ga.history["generation"], ga.history["avg_cost"], 'r--', linewidth=1, alpha=0.6, label='Custo médio')
ax2.set_xlabel('Geração')
ax2.set_ylabel('Custo')
ax2.set_title(f'Algoritmo Genético (Melhor: {ga.best_cost:.2f})')
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()

print(f"\nResumo:")
print(f"  Têmpera Simulada: {sa.best_cost:.2f}")
print(f"  Algoritmo Genético: {ga.best_cost:.2f}")
print(f"  Vencedor: {'AG' if ga.best_cost < sa.best_cost else 'SA'}")

## 6. Análise de Sensibilidade

Experimente variando parâmetros para entender o impacto na convergência.

In [ ]:
# Teste com diferentes temperaturas iniciais
temperatures = [100, 500, 1000, 5000]
results_temp = []

for temp in temperatures:
    sa_test = SimulatedAnnealing(tsp, initial_temp=temp, cooling_rate=0.95, max_iterations=500)
    _, cost = sa_test.run(verbose=False)
    results_temp.append(cost)
    print(f"Temp inicial {temp:5d}: custo = {cost:.2f}")

print()

# Teste com diferentes taxas de mutação
mutation_rates = [0.01, 0.05, 0.1, 0.2]
results_mut = []

for mut_rate in mutation_rates:
    ga_test = GeneticAlgorithm(tsp, population_size=30, mutation_rate=mut_rate, max_generations=100)
    _, cost = ga_test.run(verbose=False)
    results_mut.append(cost)
    print(f"Taxa de mutação {mut_rate:.2f}: custo = {cost:.2f}")

## Próximos passos

1. **Testar com instâncias maiores** (20, 50, 100 cidades)
2. **Variar operadores de cruzamento e mutação**
3. **Implementar elitismo** no AG
4. **Comparar com bounds teóricos** (lower bounds do TSP)
5. **Parallelizar** a execução de múltiplas instâncias